# **Fine Tuned Model Testing**

In [2]:
# !pip install -U bitsandbytes transformers peft accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 38.6 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

In [4]:
# 1. Base model ID
base_model_id = "unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit"

In [5]:
# 2. Path to uploaded dataset aka fine-tuned adapter
# adapter_path = "/kaggle/input/Nyaya-Llama-MTech-Adapter"
adapter_path = "/kaggle/input/datasets/rehanfargose/nyaya-llama-mtech-adapter"

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [7]:
# 3. Load base model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config, 
    device_map="auto"
)

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:250: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [8]:
model = PeftModel.from_pretrained(model, adapter_path)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

In [9]:
print("Adapter successfully loaded from private dataset!")

Adapter successfully loaded from private dataset!


In [11]:
case_facts = "The appellant, acting as a Power of Attorney holder for the property owners, leased a commercial premises to Respondent No. 1 (Bala Venkatram) in May 2007 for running a business named 'Best Mark Super Market'. Following a default in rent and a change in the shop's name to 'Amutham Super Market', the appellant discovered that Respondent No. 1 had effectively handed over the premises to Respondent No. 2 (Shahu Hameed). The appellant filed an eviction petition on grounds of sub-letting and arrears of rent. While the respondents claimed an oral partnership existed between them, the Rent Control Appellate Authority granted eviction, which was later set aside by the High Court. The Supreme Court was tasked with determining if a genuine partnership existed or if it was an unauthorized sub-letting."

In [12]:
case_evidence = ['Rental Agreement dated 23.05.2007 between the appellant and respondent no. 1.', 'Power of Attorney dated 01.11.2006 executed in favor of the appellant.', 'Sales Tax Certificate and Registration in the name of respondent no. 2.', 'Shop License issued in the name of respondent no. 2.', 'Bank accounts of the firm showing respondent no. 2 as the sole owner.', 'Cross-examination of Respondent No. 1 where he admitted handing over the premises to respondent no. 2.', 'Legal notice issued by the landlady to the original tenant.', 'Absence of any written partnership deed or document proving a joint business venture.']


In [14]:
case_ipc = ['Tamil Nadu Buildings (Lease and Rent Control) Act, 1960', 'Section 10(2)(i) (Eviction for arrears of rent)', 'Section 10(2)(ii)(a)(b) (Eviction for sub-letting/transfer of rights)', 'Section 10(2)(iii) (Eviction for wastage or material alteration)', 'Section 2(6) (Definition of Landlord)']
    

In [15]:
prompt = f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a legal analyst. Scrutinize the provided EVIDENCE against the established FACTS.
Identify contradictions, timeline gaps, or logical inconsistencies.
Be concise and objective.<|eot_id|>

<|start_header_id|>user<|end_header_id|>
### ESTABLISHED FACTS:
{case_facts}

### DISPUTED EVIDENCE:
{case_evidence}

### ANALYSIS:
<|assistant|><|end_header_id|>
"""

In [16]:
# 4. Generate Analysis
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=300, 
        temperature=0.1, 
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )

In [17]:
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

<|begin_of_text|>
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a legal analyst. Scrutinize the provided EVIDENCE against the established FACTS.
Identify contradictions, timeline gaps, or logical inconsistencies.
Be concise and objective.<|eot_id|>

<|start_header_id|>user<|end_header_id|>
### ESTABLISHED FACTS:
The appellant, acting as a Power of Attorney holder for the property owners, leased a commercial premises to Respondent No. 1 (Bala Venkatram) in May 2007 for running a business named 'Best Mark Super Market'. Following a default in rent and a change in the shop's name to 'Amutham Super Market', the appellant discovered that Respondent No. 1 had effectively handed over the premises to Respondent No. 2 (Shahu Hameed). The appellant filed an eviction petition on grounds of sub-letting and arrears of rent. While the respondents claimed an oral partnership existed between them, the Rent Control Appellate Authority granted eviction, which was later set aside by